In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 

from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
from tqdm.auto import tqdm
import librosa
import warnings
warnings.filterwarnings('ignore')

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


# Make stimuli for 2024 mono same vs different sex distractor control experiment - now where cue level does not match target level 

Also use expanded set of words and foregrounds 
___
## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-distractor same sex
- 1-distractor different sex
___
Will be using foregrounds and cues from manifest:
`/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl`
Stim will be saved at `/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024`    
Manifest to been copied to this directory as `screened_full_stim_manifest_w_fnames_transcripts_and_f0.pdpkl` 




## Load and prep manifests

#### Get max n targets from set of foregrounds

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024')
stim_out_path.mkdir(parents=True, exist_ok=True)


# manifests = pd.read_pickle('/om/user/imgriff/datasets/human_distractor_sex_2024/full_stim_manifest_w_fnames.pdpkl')
# manifest = pd.read_pickle("/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl")
manifest = pd.read_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl")


#### remove columns from df we don't need and format for experiment 

These include old src_fn, and the second distractor info (used for symmetric experiments we don't need here)

In [3]:
### Remove columns with _x, _y, or distractor_2 in the name 
cols_to_keep = [col for col in manifest.columns if '_x' not in col and '_y' not in col and 'distractor_2' not in col and 'in_s' not in col and 'split' not in col and 'distractor_orig_df_ix' not in col]
manifest = manifest[cols_to_keep].copy()
manifest['distractor_client_id'] = manifest['distractor_client_id'].apply(lambda x: x[0])
manifest['distractor_corpus'] = manifest['distractor_corpus'].apply(lambda x: x[0])
manifest['sex_cond'] = manifest['sex_cond'].str.title()
# manifest 

## Make and write audio for human experiment 

Will use same set of curated unique words as in SWC and Popham style experiments 

### Prep condition list

In [4]:
stim_out_path

PosixPath('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024')

In [5]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

distractor_conditions = ['Same',
              'Different']
target_conditions = ['female', 'male']

condition_pairs = list(itertools.product(target_conditions, distractor_conditions, snrs))
condition_pairs.append(('female', 'SILENCE', 'inf'))
condition_pairs.append(('male', 'SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}
print(cond_ix_dict)
print(len(cond_ix_dict))
out_name = stim_out_path / "human_distractor_sex_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(cond_ix_dict, f)


{0: ('female', 'Same', -9), 1: ('female', 'Same', -6), 2: ('female', 'Same', -3), 3: ('female', 'Same', 0), 4: ('female', 'Same', 3), 5: ('female', 'Different', -9), 6: ('female', 'Different', -6), 7: ('female', 'Different', -3), 8: ('female', 'Different', 0), 9: ('female', 'Different', 3), 10: ('male', 'Same', -9), 11: ('male', 'Same', -6), 12: ('male', 'Same', -3), 13: ('male', 'Same', 0), 14: ('male', 'Same', 3), 15: ('male', 'Different', -9), 16: ('male', 'Different', -6), 17: ('male', 'Different', -3), 18: ('male', 'Different', 0), 19: ('male', 'Different', 3), 20: ('female', 'SILENCE', 'inf'), 21: ('male', 'SILENCE', 'inf')}
22


## Make stimuli 

In [6]:
unique_words = sorted(manifest.word.unique())

In [7]:
# manifest[(manifest.gender == 'female') & (manifest.sex_cond == "Same")]

### Re worked experiment to not over think it - save all possible combinations and use prolific to randomize subset selection 

In [8]:
def get_avg_f0(sig, SR, fmin=80, fmax=300):
    # get talker f0 - bound to range of human voice 
    target_f0, voice_flag, voice_probs = librosa.pyin(sig, fmin=fmin, fmax=fmax, sr=SR, fill_na=np.nan)
    target_f0 = target_f0[voice_flag]        
    avg_target_f0 = np.nanmean(target_f0)
    return avg_target_f0

In [9]:
# np.random.seed(1) # set seed 

# out_dir = stim_out_path / 'sounds'

# SR = 44100
# SPL = 60 # in dB SPL 
# # timing in seconds 
# trial_dur = 4.5
# signal_dur = 2
# isi = 0.5
# win_dur = 10 # ms 
# new_rms = 0.02 # is 60dB in  amplitude e.g 20e-6pa * 10**(60/20)
# mixture_onset = int((isi + signal_dur) * SR) # in frames


# for cond_id, (target_sex, sex_cond, snr) in cond_ix_dict.items():
#     stim_out_dir = out_dir / f"condition_{cond_id:02}"
#     stim_out_dir.mkdir(parents=True, exist_ok=True)
#     print(f"Saving stimuli to {stim_out_dir.as_posix()}")
#     cond_manifest = manifest[(manifest.gender == target_sex) & (manifest.sex_cond == sex_cond)].copy()
    
#     # sort cond_manifest by word 
#     cond_manifest = cond_manifest.sort_values("word").reset_index(drop=True)
#     assert (cond_manifest.word == unique_words).all() , "Missing words in condition manifest"

#     ## Write out trials
#     fname_array = cond_manifest[["excerpt_cue_src_fn", "excerpt_src_fn", "excerpt_distractor_1_src_fn"]].values

#     for ix, (cue_fn, target_fn, distractor_fn) in enumerate(tqdm(fname_array, leave=False)):
#         # init output signal 
#         trial_signal = np.zeros((int(SR * trial_dur)),dtype=np.float32)
#         # load already cut/resampled/windowed signals 
#         cue_signal, cue_sr = librosa.load(cue_fn, sr=SR)
#         target_signal, target_sr = librosa.load(target_fn, sr=SR)
#         distractor_signal, dist_sr = librosa.load(distractor_fn, sr=SR)

#         # get f0 
#         cue_f0 = get_avg_f0(cue_signal, SR)
#         target_f0 = get_avg_f0(target_signal, SR)
#         distractor_f0 = get_avg_f0(distractor_signal, SR)
#         cond_manifest.loc[ix, 'cue_f0'] = cue_f0
#         cond_manifest.loc[ix, 'target_f0'] = target_f0
#         cond_manifest.loc[ix, 'distractor_f0'] = distractor_f0

#         cue_signal = util_audio.set_dBSPL(cue_signal, SPL)
#         target_signal = util_audio.set_dBSPL(target_signal, SPL)
#         distractor_signal = util_audio.set_dBSPL(distractor_signal, SPL)

#         if snr == 'inf':
#             mixture = target_signal 
#         else:
#             mixture = util_audio.combine_signal_and_noise(target_signal, distractor_signal, snr)
#         mixture = util_audio.ramp_hann(mixture, win_dur, samplerate=SR)
#         mixture = util_audio.set_dBSPL(mixture, SPL)
        
#         # set cue to level - IS UPDATE: NOT SCALING TO MATCH TARGET LEVEL
#         cue_signal = util_audio.ramp_hann(cue_signal, win_dur, samplerate=SR)
#         cue_signal = util_audio.set_dBSPL(cue_signal, SPL)

#         # add parts to trial signal
#         trial_signal[ : len(cue_signal)] += cue_signal  
#         trial_signal[mixture_onset : ] += mixture
#         trial_signal = util_audio.set_dBSPL(trial_signal, SPL)

#         # save trial signal
#         # f string with digint padded to 3 places
#         out_name = stim_out_dir / f'{ix:03}.wav'
#         sf.write(out_name, trial_signal, samplerate=SR)
#         # break
    
#     ## Write out the manifest for this condition
#     cond_manifest_out_name = stim_out_dir / "condition_manifest.pdpkl"
#     cond_manifest.to_pickle(cond_manifest_out_name)
#     # break



In [10]:
# Assuming all necessary imports and variables are already defined above this point



In [14]:
from concurrent.futures import ProcessPoolExecutor
# np.random.seed(1) # set seed 

out_dir = stim_out_path / 'sounds'
SR = 44100
SPL = 60 # in dB SPL 
# timing in seconds 
trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
mixture_onset = int((isi + signal_dur) * SR) # in frames



def process_trial(args):
    cue_fn, target_fn, distractor_fn, snr, stim_out_dir, ix, SR, SPL, trial_dur, mixture_onset, win_dur = args
    trial_signal = np.zeros((int(SR * trial_dur)), dtype=np.float32)
    cue_signal, cue_sr = librosa.load(cue_fn, sr=SR)
    target_signal, target_sr = librosa.load(target_fn, sr=SR)
    distractor_signal, dist_sr = librosa.load(distractor_fn, sr=SR)

    cue_f0 = get_avg_f0(cue_signal, SR)
    target_f0 = get_avg_f0(target_signal, SR)
    distractor_f0 = get_avg_f0(distractor_signal, SR)

    cue_signal = util_audio.set_dBSPL(cue_signal, SPL)
    target_signal = util_audio.set_dBSPL(target_signal, SPL)
    distractor_signal = util_audio.set_dBSPL(distractor_signal, SPL)

    if snr == 'inf':
        mixture = target_signal
    else:
        mixture = util_audio.combine_signal_and_noise(target_signal, distractor_signal, snr)
    mixture = util_audio.ramp_hann(mixture, win_dur, samplerate=SR)
    mixture = util_audio.set_dBSPL(mixture, SPL)

    cue_signal = util_audio.ramp_hann(cue_signal, win_dur, samplerate=SR)
    cue_signal = util_audio.set_dBSPL(cue_signal, SPL)

    trial_signal[:len(cue_signal)] += cue_signal
    trial_signal[mixture_onset:] += mixture
    trial_signal = util_audio.set_dBSPL(trial_signal, SPL)

    out_name = stim_out_dir / f'{ix:03}.wav'
    sf.write(out_name, trial_signal, samplerate=SR)

    return ix, cue_f0, target_f0, distractor_f0

for cond_id, (target_sex, sex_cond, snr) in cond_ix_dict.items():
    if snr != 'inf':
        continue 
    stim_out_dir = out_dir / f"condition_{cond_id:02}"
    stim_out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving stimuli to {stim_out_dir.as_posix()}")
    if sex_cond == "SILENCE":
        # same targets, distractor choice is irrelevant
        cond_manifest = manifest[(manifest.gender == target_sex) & (manifest.sex_cond == "Same")].copy()
    else:
        cond_manifest = manifest[(manifest.gender == target_sex) & (manifest.sex_cond == sex_cond)].copy()
    
    # sort cond_manifest by word 
    cond_manifest = cond_manifest.sort_values("word").reset_index(drop=True)
    assert (cond_manifest.word == unique_words).all() , "Missing words in condition manifest"

    ## Write out trials
    fname_array = cond_manifest[["excerpt_cue_src_fn", "excerpt_src_fn", "excerpt_distractor_1_src_fn"]].values

    # Prepare arguments for each trial
    args_list = [(cue_fn, target_fn, distractor_fn, snr, stim_out_dir, ix, SR, SPL, trial_dur, mixture_onset, win_dur)
                for ix, (cue_fn, target_fn, distractor_fn) in enumerate(fname_array)]

    # Process trials in parallel
    with ProcessPoolExecutor() as executor:
        results = list(tqdm(executor.map(process_trial, args_list), total=len(args_list)))

    # Update cond_manifest with results
    for ix, cue_f0, target_f0, distractor_f0 in results:
        cond_manifest.loc[ix, 'cue_f0'] = cue_f0
        cond_manifest.loc[ix, 'target_f0'] = target_f0
        cond_manifest.loc[ix, 'distractor_f0'] = distractor_f0

    # Write out the manifest for this condition
    cond_manifest_out_name = stim_out_dir / "condition_manifest.pdpkl"
    cond_manifest.to_pickle(cond_manifest_out_name)

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024/sounds/condition_20


  0%|          | 0/488 [00:00<?, ?it/s]

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024/sounds/condition_21


  0%|          | 0/488 [00:00<?, ?it/s]

In [13]:
sex_cond

'SILENCE'

In [ ]:
ipd.display(ipd.Audio(cue_signal, rate=SR, normalize=False))
ipd.display(ipd.Audio(target_signal, rate=SR, normalize=False))
ipd.display(ipd.Audio(distractor_signal, rate=SR, normalize=False))
ipd.display(ipd.Audio(trial_signal, rate=SR, normalize=False))

In [15]:
out_dir

PosixPath('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024/sounds')

In [16]:
stim_out_path

PosixPath('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024')

In [26]:
## Merge condition all_manifests for later analysis 
all_manifests = []
for manifest_path in out_dir.rglob('condition_manifest.pdpkl'):
    df = pd.read_pickle(manifest_path)
    df['condition_dir'] = manifest_path.parent.name
    all_manifests.append(df)

final_manifest = pd.concat(all_manifests, axis=0, ignore_index=True)

# save 
final_manifest_out_name = stim_out_path / "full_expmt_stim_manifest_w_transcripts_and_f0.pdpkl"
final_manifest.to_pickle(final_manifest_out_name)




### Make target word key 


In [28]:
all([all(final_manifest.word.unique() == all_manifests[i].word.values) for i in range(len(all_manifests))])


True

In [41]:
stim_out_path

PosixPath('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024')

In [30]:
import json
import pickle 


exmpt_word_dict = {i:w for i,w in enumerate(sorted(final_manifest.word.unique()))}

# save as pickle
out_name = stim_out_path /  "human_distractor_sex_word_key_488.pkl"
with open(out_name, 'wb') as f:
    pickle.dump(exmpt_word_dict, f)



words = list(exmpt_word_dict.values())
# save as json 
out_name = stim_out_path /  "human_distractor_sex_word_key_488.js"
with open(out_name, 'w') as f:
    json.dump({"dictionary":words}, f)



In [31]:
len(exmpt_word_dict)

488

In [ ]:
## Make word Key .js from pickle of word dict
import IPython.display as ipd


In [ ]:
trial_ix = 10
print(f"word: {exmpt_word_dict[trial_ix]}")
ipd.Audio(f"/om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_04/{trial_ix:03}.wav")

## Make sure stimuli generation worked 

In [33]:
out_dir

PosixPath('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024/sounds')

In [34]:
paths = list(Path('/om/user/imgriff/datasets/human_distractor_sex_no_cue_level_match_2024/sounds/').rglob('*/*.wav'))

In [35]:
import librosa

In [37]:
bad_paths = []
for path in tqdm(paths):
    try:
        librosa.get_duration(filename=path)
    except Exception as e:
        print(e)
        bad_paths.append(path)

  0%|          | 0/10736 [00:00<?, ?it/s]